# DFD Staffing Capacity & Operational Risk Analysis 2

#### Questions

#### Importing Libraries

In [155]:
import pandas as pd     
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

#### Importing Input File

In [156]:
# File path
file_path = "C:/Users/AmarachiOny/Desktop/DFD_Projects/Python/DFD_Staffing_Analysis/input/Staffing_Tracker_2023_Copy_Sheet1.csv"

# Load the CSV file
df = pd.read_csv(file_path)

# Printing the column names
print("Column names:")
print(df.columns.tolist())

Column names:
['Date', 'Shift', 'Total Shift FTEs', 'Current Shift FTEs', 'Total @ Work', 'Total Absences', 'VAC', 'SICK', 'Light Duty', 'PPL', 'FMLA PTD', 'COVID', 'USAR', 'TNG', 'MIL', 'Other', 'REASS', 'Hireback Total', 'FF Hireback', 'Driver Hireback', 'Captain Hireback', 'BC Hireback', 'Total Vehicles', 'App 4 Staffed', 'APP OOS', 'Support OOS', 'Total Ops FTEs', 'OPS VAC    ###', 'Notes', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31']


#### Dropping Columns

In [157]:
df = df.drop(columns=['Notes', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31'])

#### Standardize Date

In [158]:
# Date Formatting
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Full date (MM/DD/YYYY)
df['Full_Date'] = df['Date'].dt.strftime('%m/%d/%Y')

# Day number with suffix (1st, 2nd, 3rd, 4th...)
def day_suffix(day):
    if 11 <= day <= 13:
        return f"{day}th"
    last_digit = day % 10
    if last_digit == 1:
        return f"{day}st"
    elif last_digit == 2:
        return f"{day}nd"
    elif last_digit == 3:
        return f"{day}rd"
    else:
        return f"{day}th"

df['Day'] = df['Date'].dt.day
df['Day_With_Suffix'] = df['Day'].apply(day_suffix)

# Month name
df['Month'] = df['Date'].dt.month_name()

# Year
df['Year'] = df['Date'].dt.year

#### Standardize Shift

In [159]:
# Shift Formatting
df['Shift'] = (
    df['Shift']
    .astype(str)
    .str.strip()
    .str.upper()
)

#### Force Numeric Columns

In [160]:
numeric_cols = [
    'Total Shift FTEs',
    'Current Shift FTEs',
    'Total @ Work',
    'Total Absences',
    'VAC',
    'SICK',
    'Light Duty',
    'PPL',
    'FMLA PTD',
    'COVID',
    'USAR',
    'TNG',
    'MIL',
    'Other',
    'REASS',
    'Hireback Total',
    'FF Hireback',
    'Driver Hireback',
    'Captain Hireback',
    'BC Hireback',
    'Total Vehicles',
    'App 4 Staffed',
    'APP OOS',
    'Support OOS',
    'Total Ops FTEs',
    'OPS VAC    ###'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

#### Derived Metrics 1: Staffing Utilization Percentage

In [161]:
# Total Shift FTEs (available slots to occupy) --> Current Shift FTEs (actuall filled slots by employees) --> Total @ Work (who showed up today)

# Thresholds:
#🟢 ≥ 85% → Healthy
#🟡 85–70% → Strained
#🔴 < 70% → Critical

# Staffing Utilization
df['Staffing_Utilization_Pct'] = (
    df['Total @ Work'] / df['Total Shift FTEs'] * 100
).replace([np.inf, -np.inf], np.nan)

# Staffing Status Flag
def staffing_flag(util):
    if util >= 85:
        return 'Healthy'
    elif util >= 70:
        return 'Strained'
    else:
        return 'Critical'

df['Staffing_Status'] = df['Staffing_Utilization_Pct'].apply(staffing_flag)

#### Derived Metrics 2: Coverage Gap

In [162]:
# Coverage Gap/ FTEs Slots to fill up
df['Coverage_Gap'] = df['Total Shift FTEs'] - df['Total @ Work'] 

#### Derived Metrics 3: Absence Burden Percentage

In [163]:
# Absence Burden Percentage = (Total Absences ÷ Total Current FTEs (Actual people who were hired for the shift and are still here))
df['Absence_Burden_Pct'] = (
    df['Total Absences'] / df['Current Shift FTEs'] * 100
).replace([np.inf, -np.inf], np.nan)

#### Derived Metrics 4: Hireback Used Flag

In [164]:
# Hireback Used Flag
df['Hireback_Used'] = np.where(df['Hireback Total'] > 0, 'Yes', 'No')

#### Final Column Order

In [165]:
final_columns = [
    'Full_Date',
    'Day_With_Suffix',
    'Month',
    'Year',
    'Shift',
    'Total Shift FTEs',
    'Current Shift FTEs',
    'Total @ Work',
    'Total Absences',
    'Coverage_Gap',
    'Staffing_Utilization_Pct',
    'Staffing_Status',
    'Absence_Burden_Pct',
    'Hireback_Used',
    'FF Hireback',
    'Driver Hireback',
    'Captain Hireback',
    'BC Hireback',
    'VAC',
    'SICK',
    'Light Duty',
    'PPL',
    'FMLA PTD',
    'MIL',
    'USAR',
    'TNG',
    'REASS',
    'APP OOS',
    'Support OOS',
    'Total Ops FTEs'
]

df_final = df[final_columns]

#### Row Count and Preview

In [166]:
# Printing the number of rows
print(f"Total number of rows: {len(df)}")

# Printing the updated column names
print("Updated column names:")
print(df.columns.tolist())

# Displaying the head
display(df.head())

Total number of rows: 1693
Updated column names:
['Date', 'Shift', 'Total Shift FTEs', 'Current Shift FTEs', 'Total @ Work', 'Total Absences', 'VAC', 'SICK', 'Light Duty', 'PPL', 'FMLA PTD', 'COVID', 'USAR', 'TNG', 'MIL', 'Other', 'REASS', 'Hireback Total', 'FF Hireback', 'Driver Hireback', 'Captain Hireback', 'BC Hireback', 'Total Vehicles', 'App 4 Staffed', 'APP OOS', 'Support OOS', 'Total Ops FTEs', 'OPS VAC    ###', 'Full_Date', 'Day', 'Day_With_Suffix', 'Month', 'Year', 'Staffing_Utilization_Pct', 'Staffing_Status', 'Coverage_Gap', 'Absence_Burden_Pct', 'Hireback_Used']


,Date,Shift,Total Shift FTEs,Current Shift FTEs,Total @ Work,Total Absences,VAC,SICK,Light Duty,PPL,...,Full_Date,Day,Day_With_Suffix,Month,Year,Staffing_Utilization_Pct,Staffing_Status,Coverage_Gap,Absence_Burden_Pct,Hireback_Used
0,2021-07-01,C,116,116,93,28,13,8,1,0,...,07/01/2021,1,1st,July,2021,80.172414,Strained,23,24.137931,Yes
1,2021-07-02,A,115,115,93,24,14,5,2,0,...,07/02/2021,2,2nd,July,2021,80.869565,Strained,22,20.869565,Yes
2,2021-07-03,C,116,116,90,31,14,9,1,0,...,07/03/2021,3,3rd,July,2021,77.586207,Strained,26,26.724138,Yes
3,2021-07-04,A,115,115,92,25,16,5,1,0,...,07/04/2021,4,4th,July,2021,80.000000,Strained,23,21.739130,Yes
4,2021-07-05,B,117,117,87,31,14,9,2,6,...,07/05/2021,5,5th,July,2021,74.358974,Strained,30,26.495726,Yes


In [167]:
# Columns
final_columns = [
    'Year', 'Coverage_Gap', 'Total Absences', 'Staffing_Utilization_Pct', 'Total @ Work',
    'FF Hireback', 'Driver Hireback', 'Captain Hireback', 'BC Hireback',
    'SICK', 'VAC', 'PPL', 'FMLA PTD', 'MIL', 'USAR', 'TNG', 'REASS', 'Light Duty'
]
df = df[final_columns]

# Convert numeric
for col in final_columns[1:]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Aggregate yearly averages
yearly = df.groupby('Year')[final_columns[1:]].mean().reset_index()

# Set style
sns.set(style="whitegrid", palette="muted", font_scale=1.0)

# Create PDF
with PdfPages("Staffing_Absence_Report.pdf") as pdf:

    # 1. Coverage Gap + Total Absences
    fig, ax = plt.subplots(figsize=(14,8))
    ax.plot(yearly['Year'], yearly['Coverage_Gap'], marker='o', label='Coverage Gap')
    ax.plot(yearly['Year'], yearly['Total Absences'], marker='^', label='Total Absences')
    ax.set_title("Coverage Gap & Total Absences Over Years")
    ax.set_xlabel("Year")
    ax.set_ylabel("Average per Shift/Day")
    ax.legend()
    ax.grid(True)
    pdf.savefig(fig)
    plt.close()

    # 2. Staffing Utilization & Personnel
    fig, ax = plt.subplots(figsize=(14,8))
    ax.plot(yearly['Year'], yearly['Staffing_Utilization_Pct'], marker='o', label='Staffing Utilization %')
    ax.plot(yearly['Year'], yearly['Total @ Work'], marker='s', label='Average Personnel @ Work')
    ax.set_title("Staffing Utilization & Personnel at Work")
    ax.set_xlabel("Year")
    ax.set_ylabel("Percentage / Count")
    ax.legend()
    ax.grid(True)
    pdf.savefig(fig)
    plt.close()

    # 3. Role-specific Hirebacks
    fig, ax = plt.subplots(figsize=(14,8))
    width = 0.2
    years = yearly['Year']
    ax.bar(years - width*1.5, yearly['FF Hireback'], width=width, label='FF Hireback')
    ax.bar(years - width/2, yearly['Driver Hireback'], width=width, label='Driver Hireback')
    ax.bar(years + width/2, yearly['Captain Hireback'], width=width, label='Captain Hireback')
    ax.bar(years + width*1.5, yearly['BC Hireback'], width=width, label='BC Hireback')
    ax.set_title("Role-specific Hirebacks Over Years")
    ax.set_xlabel("Year")
    ax.set_ylabel("Average Hirebacks")
    ax.legend()
    pdf.savefig(fig)
    plt.close()

    # 4. Detailed Absence Types
    absence_cols = ['SICK', 'VAC', 'PPL', 'FMLA PTD', 'MIL', 'USAR', 'TNG', 'REASS', 'Light Duty']
    fig, ax = plt.subplots(figsize=(14,8))
    for col in absence_cols:
        ax.plot(yearly['Year'], yearly[col], marker='o', label=col)
    ax.set_title("Breakdown of Absence Types Over Years")
    ax.set_xlabel("Year")
    ax.set_ylabel("Average Absences per Shift/Day")
    ax.legend(loc='upper left', bbox_to_anchor=(1,1))
    ax.grid(True)
    pdf.savefig(fig)
    plt.close()

    # 5. Summary Table (without Total Hirebacks)
    fig, ax = plt.subplots(figsize=(14,4))
    ax.axis('off')
    ax.set_title("Summary Table of Key Metrics")

    # Keep only core metrics
    table_data = yearly[['Year', 'Coverage_Gap', 'Total Absences', 'Staffing_Utilization_Pct', 'Total @ Work']]

    # Create the table
    table = ax.table(cellText=table_data.values,
                     colLabels=table_data.columns,
                     loc='center',
                     cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.auto_set_column_width(col=list(range(len(table_data.columns))))
    pdf.savefig(fig)
    plt.close()

print("PDF report created: Staffing_Absence_Report.pdf")

PDF report created: Staffing_Absence_Report.pdf
